## Synthetic Data Enrichment for PaySim Mobile Money Transactions

### Background

The PaySim Mobile Money dataset is a synthetic dataset mainly used for fraud detection research in mobile financial transactions. While it simulates realistic transaction flows and fraud patterns, it lacks important business and operational context needed for real-world analytics. Because of these gaps, the dataset cannot fully support business-focused analysis like  operational monitoring, or executive reporting.

To address this, synthetic data enrichment is applied to extend the dataset into a more realistic payment ecosystem, enabling both fraud detection and business intelligence analysis within a single framework.


### Data Enrichment Design 

To extend the original PaySim dataset, additional fields and two new dimension tables were introduced to better simulate a real-world mobile payment system.

Added to Transactions Table

The following fields were either newly created or derived from transformations of the original dataset:

`transaction_id:` Unique transaction identifier <br>
`timestamp:` Exact date and time of transaction <br>
`payment_method:` Standardized transaction category (transfer, payment, etc.) <br>
`gateway:` Payment processor used <br>
`transaction_status:` Outcome of transaction (success, failed) <br>
`failure_reason:` Reason for failed transactions <br>
`transaction_fee:` Simulated fee charged per transaction <br>
`gateway_cost:` Estimated system processing cost <br>
`processing_time_seconds:` Simulated processing duration <br>
`country:` Origin country of transaction <br>
`device_type:` Device used for transaction <br>
`risk_score:` Fraud probability from XGBoost model <br>
`fraud_flag:` Derived binary fraud indicator based on risk_score <br>
`chargeback_flag:` Indicates if a chargeback occurred <br>

New Table: Customers

A new dimension table was created to enable customer-level analysis:

`customer_id:` Unique customer identifier (mapped from nameOrig) <br>
`home_country:` Customer’s country of residence <br>
`primary_device:` Most frequently used device <br>
`customer_signup_date:` Account creation date <br>

New Table: Merchants

A new dimension table was introduced for merchant-level insights:

`merchant_id:` Unique merchant identifier (mapped from nameDest) <br>
`merchant_country:` Country of operation <br>
`merchant_category:` Business classification <br>
`merchant_onboarding_date:` Date merchant joined the platform <br>


*Note: These are synthetics datasets the logic behind populating these datasets may not simulate actual behaviour of actual transactions. The only pupose for this is to create a Power BI template for Financial Performance and Operations Monitoring*

### Methodology/Approach

#### I. Basic Data Enrichment/Transformation

##### *Import libraries*

In [1]:
from dotenv import load_dotenv
from sqlalchemy import create_engine
from datetime import datetime, timedelta

import os
import pandas as pd
import numpy as np
import uuid

In [2]:
# Load environment variables
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Load dataset
query = "SELECT * FROM labeled_dataset"
df = pd.read_sql(query, engine)

In [3]:
paysim_df = df.copy()
paysim_df.head()

,step,type,amount,nameOrig,nameDest,isFraud,hour_of_day,day_of_week,FraudProbability,isFlaggedFraud
0,1,PAYMENT,6.93,C730584984,M1276666395,0,1,0,0.000002,0
1,1,PAYMENT,13.54,C1172417096,M314966354,0,1,0,0.000002,0
2,1,PAYMENT,15.06,C1730337646,M418513504,0,1,0,0.000002,0
3,1,PAYMENT,53.35,C1908999587,M816804727,0,1,0,0.000002,0
4,1,PAYMENT,79.26,C2034975583,M955443582,0,1,0,0.000002,0


##### *Standardize colum names and values*

In [4]:
# Rename columns
paysim_df = paysim_df.rename(columns={
    "type": "payment_method",
    "nameOrig": "customer_id",
    "nameDest": "destination_id",
    "isFraud": "is_fraud",
    "FraudProbability": "risk_score",
    "isFlaggedFraud": "fraud_flag"
})

In [5]:
# Pad customer and merchant IDs
cols = ["customer_id", "destination_id"]

for col in cols:
    paysim_df[col] = paysim_df[col].str.pad(
        width=paysim_df[col].str.len().max(),
        side="right",
        fillchar="X"
    )

##### *Timestamp, week, and transaction id columns*

In [6]:
# Simulated timestamp
base_timestamp = pd.Timestamp("2025-01-01")

n = len(paysim_df)

paysim_df["timestamp"] = (
    base_timestamp
    + pd.to_timedelta(paysim_df["step"], unit="h")
    + pd.to_timedelta(np.random.randint(0, 60, n), unit="m")
    + pd.to_timedelta(np.random.randint(0, 60, n), unit="s")
    + pd.to_timedelta(np.random.randint(0, 1000, n), unit="ms")
)

# Add week of month column
paysim_df["week_of_month"] = ((paysim_df["timestamp"].dt.day - 1) // 7) + 1

# Add unique transaction_id
paysim_df["transaction_id"] = pd.Series([uuid.uuid4() for _ in range(n)], dtype="string")

##### *View Results*

In [7]:
paysim_df.head()

,step,payment_method,amount,customer_id,destination_id,is_fraud,hour_of_day,day_of_week,risk_score,fraud_flag,timestamp,week_of_month,transaction_id
0,1,PAYMENT,6.93,C730584984X,M1276666395,0,1,0,0.000002,0,2025-01-01 01:48:18.869,1,a6bc9512-d947-4e90-9782-75fd657d741d
1,1,PAYMENT,13.54,C1172417096,M314966354X,0,1,0,0.000002,0,2025-01-01 01:43:06.013,1,0ede23d4-13c3-4cf8-86e7-7944d13f85c6
2,1,PAYMENT,15.06,C1730337646,M418513504X,0,1,0,0.000002,0,2025-01-01 01:15:40.135,1,c209745d-68f3-406e-a2fb-69502705fcfb
3,1,PAYMENT,53.35,C1908999587,M816804727X,0,1,0,0.000002,0,2025-01-01 01:21:31.173,1,8e77eef3-98da-4ea8-88cb-fd3a8872e2c4
4,1,PAYMENT,79.26,C2034975583,M955443582X,0,1,0,0.000002,0,2025-01-01 01:50:49.800,1,53ef2682-beb8-4c69-8403-ec63cf3b676a


#### II. Customer Dimension Table

In [8]:
# Customer base
customers = paysim_df["customer_id"].unique()

start_date = pd.Timestamp("2022-12-01")
end_date = pd.Timestamp("2025-01-01")
date_span_days = (end_date - start_date).days

n_customers = len(customers)

# Generate customer attributes
customer_df = pd.DataFrame({
    "customer_id": customers,

    "home_country": np.random.choice(
        ["PH", "SG", "MY", "ID"],
        size=n_customers,
        p=[0.40, 0.25, 0.20, 0.15]
    ),

    "primary_device": np.random.choice(
        ["Mobile", "Desktop", "Tablet"],
        size=n_customers,
        p=[0.70, 0.20, 0.10]
    ),

    "customer_signup_date": start_date + pd.to_timedelta(
        np.random.randint(0, date_span_days, n_customers),
        unit="D"
    )
})

##### *View Results*

In [9]:
customer_df.head()

,customer_id,home_country,primary_device,customer_signup_date
0,C730584984X,MY,Tablet,2024-07-24
1,C1172417096,PH,Mobile,2024-09-06
2,C1730337646,PH,Mobile,2024-09-13
3,C1908999587,ID,Desktop,2023-06-04
4,C2034975583,MY,Mobile,2024-01-08


#### III. Merchant Dimension Table

In [10]:
# Merchant base
merchants = (
    paysim_df.loc[
        paysim_df["destination_id"].str.startswith("M"),
        "destination_id"
    ].unique()
)

n_merchants = len(merchants)

start_date = pd.Timestamp("2022-01-01")
end_date = pd.Timestamp("2025-01-01")
date_span_days = (end_date - start_date).days

# Build merchant dimension table
merchant_df = pd.DataFrame({
    "merchant_id": merchants,

    "merchant_country": np.random.choice(
        ["PH", "SG", "MY", "ID"],
        size=n_merchants,
        p=[0.40, 0.25, 0.20, 0.15]
    ),

    "merchant_category": np.random.choice(
        ["E-commerce", "Retail", "Travel", "SaaS", "Food Delivery"],
        size=n_merchants,
        p=[0.35, 0.25, 0.10, 0.15, 0.15]
    ),

    "merchant_onboarding_date": start_date + pd.to_timedelta(
        np.random.randint(0, date_span_days, n_merchants),
        unit="D"
    )
})

##### *View Results*

In [11]:
merchant_df.head()

,merchant_id,merchant_country,merchant_category,merchant_onboarding_date
0,M1276666395,ID,E-commerce,2023-01-20
1,M314966354X,MY,E-commerce,2023-06-20
2,M418513504X,PH,Food Delivery,2023-10-25
3,M816804727X,ID,E-commerce,2024-06-29
4,M955443582X,PH,SaaS,2022-07-19


#### IV. Synthetic Data Enrichment

##### *Gateway Assignment and Associated Costs*

In [12]:
gateway_options = np.array(["Gateway A", "Gateway B", "Gateway C", "Gateway D"])

gateway_cost_rate = {
    "Gateway A": 0.0020,
    "Gateway B": 0.0028,
    "Gateway C": 0.0030,
    "Gateway D": 0.0050,
}

week_gateway_probs = {
    1: [0.35, 0.20, 0.25, 0.20],
    2: [0.60, 0.30, 0.05, 0.05],
    3: [0.10, 0.10, 0.20, 0.60],
    4: [0.35, 0.20, 0.25, 0.20]
}

default_probs = np.array([0.35, 0.20, 0.25, 0.20])

# Gateway Assignement
week_probs_matrix = np.vstack(
    paysim_df["week_of_month"].map(
        lambda w: week_gateway_probs.get(w, default_probs)
    )
)

rand_vals = np.random.rand(len(paysim_df), 1)
cum_probs = np.cumsum(week_probs_matrix, axis=1)

paysim_df["gateway"] = gateway_options[
    (rand_vals <= cum_probs).argmax(axis=1)
]

# Gateway Rate Calculation
paysim_df["gateway_rate"] = paysim_df["gateway"].map(gateway_cost_rate)

# Gateway Cost Calculation
paysim_df["gateway_cost"] = (
    paysim_df["amount"] * paysim_df["gateway_rate"]
).round(2)

##### *Transaction Status*

In [13]:
# Gateway risk rates
gateway_failure_rate = {
    "Gateway A": 0.01,
    "Gateway B": 0.02,
    "Gateway C": 0.03,
    "Gateway D": 0.05
}

# Hard fail rule
hard_fail = paysim_df["risk_score"] >= 0.85

# Gateway risk
gateway_prob = paysim_df["gateway"].map(gateway_failure_rate).fillna(0.02)

# Risk score effect
risk_prob = np.where(
    (paysim_df["risk_score"] > 0.5) & (paysim_df["risk_score"] < 0.85),
    0.25,
    0.0
)

# Time risk
time_prob = np.where(paysim_df["hour_of_day"].between(0, 8), 0.015, 0.0)

# Amount risk
amount_prob = np.where(paysim_df["amount"] >= 100000, 0.01, 0.0)

# Combined failure probability
failure_prob = 1 - (
    (1 - gateway_prob) *
    (1 - risk_prob) *
    (1 - time_prob) *
    (1 - amount_prob)
)

# Hard fail override
failure_prob = np.where(hard_fail, 1.0, failure_prob)

# Transaction outcome simulation
rng = np.random.default_rng(42)

paysim_df["transaction_status"] = np.where(
    rng.random(len(paysim_df)) < failure_prob,
    "Failed",
    "Completed"
)

##### *Reasons for Failure*

In [14]:
paysim_df["failure_reason"] = None

failed = paysim_df["transaction_status"] == "Failed"

# Fraud risk
paysim_df.loc[
    failed & (paysim_df["risk_score"] >= 0.85),
    "failure_reason"
] = "Suspected Fraud"

# Gateway / time issue
paysim_df.loc[
    failed & paysim_df["failure_reason"].isna() &
    (
        paysim_df["gateway"].isin(["Gateway C", "Gateway D"]) |
        paysim_df["hour_of_day"].between(0, 8)
    ),
    "failure_reason"
] = "Gateway Failure"

# CVV mismatch
paysim_df.loc[
    failed & paysim_df["failure_reason"].isna() &
    paysim_df["payment_method"].isin(["PAYMENT", "DEBIT"]) &
    (paysim_df["amount"] <= 50000),
    "failure_reason"
] = "CVV Mismatch"

# Funds issue
paysim_df.loc[
    failed & paysim_df["failure_reason"].isna() &
    paysim_df["payment_method"].isin(["CASH_OUT", "DEBIT", "PAYMENT"]),
    "failure_reason"
] = "Insufficient Funds"

# Default
paysim_df.loc[
    failed & paysim_df["failure_reason"].isna(),
    "failure_reason"
] = "Bank Declined"

##### *Estimated processing time*

In [15]:
rng = np.random.default_rng(42)

base_time = np.select(
    [
        paysim_df["payment_method"] == "PAYMENT",
        paysim_df["payment_method"] == "TRANSFER",
        paysim_df["payment_method"] == "CASH_OUT",
        paysim_df["payment_method"] == "CASH_IN",
        paysim_df["payment_method"] == "DEBIT"
    ],
    [
        rng.normal(4, 1, len(paysim_df)),    # PAYMENT
        rng.normal(30, 10, len(paysim_df)),  # TRANSFER
        rng.normal(10, 3, len(paysim_df)),   # CASH_OUT
        rng.normal(2, 0.5, len(paysim_df)),  # CASH_IN
        rng.normal(5, 1.5, len(paysim_df))   # DEBIT
    ],
    default=rng.normal(6, 2, len(paysim_df))
)

gateway_penalty = np.select(
    [
        paysim_df["gateway"] == "Gateway A",
        paysim_df["gateway"] == "Gateway B",
        paysim_df["gateway"] == "Gateway C",
        paysim_df["gateway"] == "Gateway D"
    ],
    [0.0, 0.5, 1.5, 3.0],
    default=1.0
)

offhour_penalty = np.where(paysim_df["hour_of_day"].between(0, 8), 1.5, 0.0)

paysim_df["processing_time_seconds"] = np.clip(
    base_time + gateway_penalty + offhour_penalty,
    0.5,
    None
).round(2)

##### *Transaction Fee*

In [16]:
# Fee configuration dictionaries
fee_rate = {
    "PAYMENT": 0.112,
    "TRANSFER": 0.045,
    "CASH_OUT": 0.075,
    "CASH_IN": 0.0,
    "DEBIT": 0.075
}

fixed_fee = {
    "PAYMENT": 15,
    "TRANSFER": 0,
    "CASH_OUT": 0,
    "CASH_IN": 0,
    "DEBIT": 10
}

# Map rates
paysim_df["fee_rate"] = paysim_df["payment_method"].map(fee_rate)
paysim_df["fixed_fee"] = paysim_df["payment_method"].map(fixed_fee)

# Base fee
base_fee = paysim_df["amount"] * paysim_df["fee_rate"] + paysim_df["fixed_fee"]

# Apply caps (optional but realistic)
min_fee = 5
max_fee = 500

base_fee = base_fee.clip(lower=min_fee, upper=max_fee)

# Only apply fee for completed transactions
paysim_df["transaction_fee"] = np.where(
    paysim_df["transaction_status"] == "Completed",
    base_fee,
    0
).round(2)

##### *Chargeback Flags*

In [17]:
paysim_df["chargeback_flag"] = 0

chargeback_condition = (
    (paysim_df["transaction_status"] == "Completed") &
    (paysim_df["payment_method"].isin(["PAYMENT", "DEBIT"]))
)

chargeback_prob = np.where(
    paysim_df["risk_score"] >= 0.85, 0.04,
    np.where(paysim_df["amount"] >= 50000, 0.02, 0.01)
)

rand_chargeback = np.random.rand(len(paysim_df))

paysim_df.loc[
    chargeback_condition & (rand_chargeback < chargeback_prob),
    "chargeback_flag"
] = 1

##### *View Results*

In [18]:
paysim_df.head()

,step,payment_method,amount,customer_id,destination_id,is_fraud,hour_of_day,day_of_week,risk_score,fraud_flag,...,gateway,gateway_rate,gateway_cost,transaction_status,failure_reason,processing_time_seconds,fee_rate,fixed_fee,transaction_fee,chargeback_flag
0,1,PAYMENT,6.93,C730584984X,M1276666395,0,1,0,0.000002,0,...,Gateway D,0.005,0.03,Completed,None,8.80,0.112,15,15.78,0
1,1,PAYMENT,13.54,C1172417096,M314966354X,0,1,0,0.000002,0,...,Gateway C,0.003,0.04,Completed,None,5.96,0.112,15,16.52,0
2,1,PAYMENT,15.06,C1730337646,M418513504X,0,1,0,0.000002,0,...,Gateway D,0.005,0.08,Completed,None,9.25,0.112,15,16.69,0
3,1,PAYMENT,53.35,C1908999587,M816804727X,0,1,0,0.000002,0,...,Gateway C,0.003,0.16,Completed,None,7.94,0.112,15,20.98,0
4,1,PAYMENT,79.26,C2034975583,M955443582X,0,1,0,0.000002,0,...,Gateway A,0.002,0.16,Completed,None,3.55,0.112,15,23.88,0


#### V. Finalize Transaction Table

##### *Merge customer table to transaction table*

In [19]:
transaction_df = paysim_df.merge(customer_df, on="customer_id", how="left")

transaction_df = transaction_df.rename(columns={
    "home_country": "country",
    "primary_device": "device_type"
})

##### *Rename columns and finalize column order*

In [20]:
transaction_df = transaction_df[[
    "transaction_id",
    "timestamp",
    "customer_id",
    "destination_id",
    "payment_method",
    "gateway",
    "amount",
    "transaction_status",
    "failure_reason",
    "transaction_fee",
    "gateway_cost",
    "processing_time_seconds",
    "country",
    "device_type",
    "is_fraud",
    "risk_score",
    "fraud_flag",
    "chargeback_flag"
]]


##### *View Results*

In [21]:
transaction_df.head()

,transaction_id,timestamp,customer_id,destination_id,payment_method,gateway,amount,transaction_status,failure_reason,transaction_fee,gateway_cost,processing_time_seconds,country,device_type,is_fraud,risk_score,fraud_flag,chargeback_flag
0,a6bc9512-d947-4e90-9782-75fd657d741d,2025-01-01 01:48:18.869,C730584984X,M1276666395,PAYMENT,Gateway D,6.93,Completed,None,15.78,0.03,8.80,MY,Tablet,0,0.000002,0,0
1,0ede23d4-13c3-4cf8-86e7-7944d13f85c6,2025-01-01 01:43:06.013,C1172417096,M314966354X,PAYMENT,Gateway C,13.54,Completed,None,16.52,0.04,5.96,PH,Mobile,0,0.000002,0,0
2,c209745d-68f3-406e-a2fb-69502705fcfb,2025-01-01 01:15:40.135,C1730337646,M418513504X,PAYMENT,Gateway D,15.06,Completed,None,16.69,0.08,9.25,PH,Mobile,0,0.000002,0,0
3,8e77eef3-98da-4ea8-88cb-fd3a8872e2c4,2025-01-01 01:21:31.173,C1908999587,M816804727X,PAYMENT,Gateway C,53.35,Completed,None,20.98,0.16,7.94,ID,Desktop,0,0.000002,0,0
4,53ef2682-beb8-4c69-8403-ec63cf3b676a,2025-01-01 01:50:49.800,C2034975583,M955443582X,PAYMENT,Gateway A,79.26,Completed,None,23.88,0.16,3.55,MY,Mobile,0,0.000002,0,0


#### VI. Save Results

##### *Save transactions table to database*

In [22]:
transaction_df.to_sql(
    name="transactions",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False,
    chunksize=10000,
    method="multi"
)

2099665

##### *Save customers table to database*

In [23]:
customer_df.to_sql(
    name="customers",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False,
    chunksize=10000,
    method="multi"
)

2098668

##### *Save merchants table to database*

In [24]:
merchant_df.to_sql(
    name="merchants",
    con=engine,
    schema="public",
    if_exists="replace",
    index=False,
    chunksize=10000,
    method="multi"
)

710477

#### *--End--*